In [62]:
%pip install pandas numpy scikit-learn catboost nbformat


   ---------------------------- ----------- 5/7 [jsonschema]
   ---------------------------------------- 7/7 [nbformat]

Note: you may need to restart the kernel to use updated packages.


In [17]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.model_selection import KFold
import plotly.express as px

In [4]:
train_df = pd.read_csv('./datasets/train.csv', sep=';', decimal=',', na_values=['None', 'nan'], low_memory=False)
test_df = pd.read_csv('./datasets/test.csv', sep=';', decimal=',', na_values=['None', 'nan'], low_memory=False)

In [5]:
train_df.select_dtypes(include=['str']).columns

Index(['dt', 'hdb_bki_total_max_limit', 'hdb_bki_total_cc_max_limit', 'gender',
       'adminarea', 'hdb_bki_total_pil_max_limit',
       'hdb_bki_active_cc_max_limit', 'city_smart_name',
       'hdb_bki_other_active_pil_outstanding',
       'dp_ewb_last_employment_position', 'hdb_bki_total_products',
       'hdb_bki_total_max_overdue_sum', 'addrref', 'bki_total_auto_cnt',
       'dp_address_unique_regions', 'hdb_bki_total_ip_max_limit',
       'hdb_bki_total_cnt', 'bki_total_oth_cnt', 'bki_total_ip_max_outstand',
       'hdb_bki_total_ip_cnt', 'hdb_bki_active_cc_max_outstand',
       'hdb_bki_total_pil_max_overdue', 'bki_total_il_max_limit',
       'bki_total_products', 'bki_active_auto_cnt', 'bki_total_max_limit',
       'hdb_bki_total_auto_max_limit', 'hdb_bki_total_micro_max_overdue',
       'bki_total_active_products', 'hdb_bki_total_active_products',
       'hdb_bki_total_micro_cnt', 'hdb_bki_active_pil_cnt',
       'period_last_act_ad', 'hdb_bki_total_cc_max_overdue',
       'hd

In [6]:
for df in [train_df, test_df]:
    for col in df.select_dtypes(include=['str']).columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='raise').astype('float64')
        except:
            df[col] = df[col].astype('str')

In [7]:
train_df.select_dtypes(include=['str']).columns

Index(['dt', 'gender', 'adminarea', 'city_smart_name',
       'dp_ewb_last_employment_position', 'addrref',
       'dp_address_unique_regions', 'period_last_act_ad'],
      dtype='str')

In [8]:
train_df['dt'] = pd.to_datetime(train_df['dt'], format='%Y-%m-%d')

# Training preprocessing

In [9]:
drop_cols = [
    'id', 'dt',
    'target', 'w',
    'dp_address_unique_regions',
    'first_salary_income',
    'period_last_act_ad'
]

X = train_df.drop(drop_cols, axis=1).copy()
y = train_df['target']
w = train_df['w']

In [10]:
cat_cols = X.select_dtypes(include=['str']).columns.tolist()
cat_cols

['gender',
 'adminarea',
 'city_smart_name',
 'dp_ewb_last_employment_position',
 'addrref']

In [11]:
for col in cat_cols:
    X[col] = X[col].fillna('missing').astype(str)

# Salary model

In [12]:
imputer_features = [c for c in X.columns if c not in [
    'salary_6to12m_avg', 'incomeValue', 'dp_ils_avg_salary_1y', 
    'dp_ils_avg_salary_2y', 'dp_ils_avg_salary_3y',
    'dp_payoutincomedata_payout_avg_3_month', 'dp_payoutincomedata_payout_avg_6_month',
    'dp_payoutincomedata_payout_avg_prev_year', 'first_salary_income'
]]

In [ ]:
def oof_impute(X, target_col, feature_cols, cat_cols_subset, n_splits=5, seed=42):
    mask_known = X[target_col].notna()
    mask_missing = ~mask_known
    
    imputed_values = pd.Series(index=X.index, dtype=float)
    
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    X_known = X.loc[mask_known, feature_cols]
    y_known = X.loc[mask_known, target_col]
    
    oof_pred = np.zeros(len(X_known))
    for tr_idx, ho_idx in kf.split(X_known):
        m = CatBoostRegressor(
            iterations=1000, depth=6, learning_rate=0.05,
            loss_function='RMSE', random_seed=seed, verbose=False
        )
        m.fit(X_known.iloc[tr_idx], y_known.iloc[tr_idx], cat_features=cat_cols_subset)
        oof_pred[ho_idx] = m.predict(X_known.iloc[ho_idx])
    
    # Для известных строк — OOF-предсказание (оценка качества imputer)
    imputed_values.loc[mask_known] = oof_pred
    
    # Для пропущенных строк — обучаем imputer на всех известных, применяем к пропущенным
    m_full = CatBoostRegressor(
        iterations=1000, depth=6, learning_rate=0.05,
        loss_function='RMSE', random_seed=seed, verbose=False
    )
    m_full.fit(X_known, y_known, cat_features=cat_cols_subset)
    imputed_values.loc[mask_missing] = m_full.predict(X.loc[mask_missing, feature_cols])
    
    # Оцениваем качество imputer на known-части (OOF)
    from sklearn.metrics import mean_absolute_error
    mae_imputer = mean_absolute_error(y_known, oof_pred)
    print(f'Imputer MAE (OOF, на известных значениях {target_col}): {mae_imputer:.0f}')
    print(f'Для сравнения, среднее {target_col}: {y_known.mean():.0f}')
    
    return imputed_values, m_full

In [19]:
cat_cols_for_imputer = [c for c in cat_cols if c in imputer_features]

imputed_salary, salary_imputer_model = oof_impute(
    X, 'salary_6to12m_avg', imputer_features, cat_cols_for_imputer
)

Imputer MAE (OOF, на известных значениях salary_6to12m_avg): 32525
Для сравнения, среднее salary_6to12m_avg: 117490


# Model (4 experts)

In [20]:
X['salary_6to12m_avg_missing'] = X['salary_6to12m_avg'].isna().astype(int)
X['salary_6to12m_avg'] = X['salary_6to12m_avg'].fillna(imputed_salary)

In [21]:
val_cutoff = pd.Timestamp('2024-06-01')

train_mask_time = (train_df['dt'] < val_cutoff).values
val_mask_time = (train_df['dt'] >= val_cutoff).values

In [22]:
X_train = X[train_mask_time].reset_index(drop=True)
y_train = y[train_mask_time].reset_index(drop=True)
w_train = w[train_mask_time].reset_index(drop=True)

X_val = X[val_mask_time].reset_index(drop=True)
y_val = y[val_mask_time].reset_index(drop=True)
w_val = w[val_mask_time].reset_index(drop=True)

In [23]:
def make_regime_4(y):
    return pd.cut(y, bins=[0, 83000, 200000, 400000, np.inf], labels=[0,1,2,3]).astype(int)

In [24]:
regime_train_4 = make_regime_4(y_train)
regime_val_4 = make_regime_4(y_val)

print(regime_train_4.value_counts().sort_index())

target
0    40123
1    15710
2     3449
3     1290
Name: count, dtype: int64


In [25]:
clf_4_tuned = CatBoostClassifier(
    learning_rate=0.051627416448269105,
    depth=8,
    l2_leaf_reg=7.9142461137166755,
    min_data_in_leaf=120,
    random_strength=3.4495138326286,
    bagging_temperature=0.4948872578136553,
    iterations=4000,
    early_stopping_rounds=50,
    task_type='GPU',
    loss_function='MultiClass',
    eval_metric='MultiClass',
    use_best_model=True,
    verbose=100,
)

clf_4_tuned.fit(
    X_train, regime_train_4,
    sample_weight=w_train, 
    cat_features=cat_cols,
    eval_set=(X_val, regime_val_4)
)

0:	learn: 1.3222282	test: 1.3145774	best: 1.3145774 (0)	total: 40.6ms	remaining: 2m 42s
100:	learn: 0.6424547	test: 0.5923459	best: 0.5923459 (100)	total: 1.64s	remaining: 1m 3s
200:	learn: 0.6073098	test: 0.5787748	best: 0.5787748 (200)	total: 2.64s	remaining: 50s
300:	learn: 0.5622091	test: 0.5664506	best: 0.5664506 (300)	total: 4.32s	remaining: 53.1s
400:	learn: 0.5270464	test: 0.5590087	best: 0.5590087 (400)	total: 6.18s	remaining: 55.4s
500:	learn: 0.5001514	test: 0.5542789	best: 0.5542789 (500)	total: 8.12s	remaining: 56.7s
600:	learn: 0.4792530	test: 0.5507992	best: 0.5507784 (595)	total: 9.99s	remaining: 56.5s
700:	learn: 0.4626222	test: 0.5482308	best: 0.5482308 (700)	total: 11.9s	remaining: 55.9s
800:	learn: 0.4480889	test: 0.5461414	best: 0.5461414 (800)	total: 14s	remaining: 56s
900:	learn: 0.4343015	test: 0.5439048	best: 0.5439048 (900)	total: 16.4s	remaining: 56.4s
1000:	learn: 0.4200968	test: 0.5411242	best: 0.5411195 (999)	total: 18.8s	remaining: 56.4s
1100:	learn: 0.40

CatBoostClassifier(bagging_temperature=0.4948872578136553, depth=8, early_stopping_rounds=50, eval_metric='MultiClass', iterations=4000, l2_leaf_reg=7.9142461137166755, learning_rate=0.051627416448269105, loss_function='MultiClass', min_data_in_leaf=120, random_strength=3.4495138326286, task_type='GPU', use_best_model=True, verbose=100)

In [26]:
regressor_configs_4 = {
    0: dict(depth=6, l2_leaf_reg=3.0, min_data_in_leaf=50, learning_rate=0.03),
    1: dict(depth=6, l2_leaf_reg=4.0, min_data_in_leaf=30, learning_rate=0.03),
    2: dict(depth=5, l2_leaf_reg=6.0, min_data_in_leaf=20, learning_rate=0.04),
    3: dict(depth=4, l2_leaf_reg=10.0, min_data_in_leaf=10, learning_rate=0.05),
}

regressors_4 = {}

for regime_id, cfg in regressor_configs_4.items():
    mask_tr = regime_train_4 == regime_id
    mask_vl = regime_val_4 == regime_id

    m = CatBoostRegressor(
        **cfg,
        eval_metric='MAE',
        iterations=4000, 
        early_stopping_rounds=50,
        verbose=100,
        task_type='GPU'
    )
    
    m.fit(
        X_train[mask_tr], y_train[mask_tr],
        sample_weight=w_train[mask_tr],
        cat_features=cat_cols,
        eval_set=(X_val[mask_vl], y_val[mask_vl])
    )
    
    regressors_4[regime_id] = m
    
    print(f'regime {regime_id}: n={mask_tr.sum()}')

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 12406.9954653	test: 16636.8818368	best: 16636.8818368 (0)	total: 86.8ms	remaining: 5m 47s
100:	learn: 10172.9076308	test: 13144.0925891	best: 13144.0925891 (100)	total: 6.01s	remaining: 3m 52s
200:	learn: 9932.3604295	test: 12800.0029867	best: 12800.0029867 (200)	total: 11.3s	remaining: 3m 33s
300:	learn: 9796.9568961	test: 12632.3240620	best: 12632.3240620 (300)	total: 16s	remaining: 3m 16s
400:	learn: 9705.8054953	test: 12538.4849729	best: 12538.4655591 (399)	total: 19.2s	remaining: 2m 52s
500:	learn: 9629.3120861	test: 12469.1446705	best: 12469.1446705 (500)	total: 22.3s	remaining: 2m 35s
600:	learn: 9563.4322533	test: 12418.8777301	best: 12418.8777301 (600)	total: 25.6s	remaining: 2m 24s
700:	learn: 9497.5607653	test: 12369.1162964	best: 12368.9826395 (699)	total: 28.9s	remaining: 2m 16s
800:	learn: 9438.3965902	test: 12325.0722419	best: 12325.0722419 (800)	total: 32.2s	remaining: 2m 8s
900:	learn: 9393.7068226	test: 12302.2019787	best: 12302.0952025 (899)	total: 35.6s	re

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 25113.7503187	test: 35517.9044970	best: 35517.9044970 (0)	total: 29.9ms	remaining: 1m 59s
100:	learn: 20915.4414174	test: 28707.9462216	best: 28707.9462216 (100)	total: 3.03s	remaining: 1m 56s
200:	learn: 20343.7311680	test: 27795.9035698	best: 27795.9035698 (200)	total: 6.01s	remaining: 1m 53s
300:	learn: 19931.3317526	test: 27353.1757070	best: 27353.1757070 (300)	total: 9.21s	remaining: 1m 53s
400:	learn: 19574.8093082	test: 27071.1265647	best: 27071.1265647 (400)	total: 12.3s	remaining: 1m 50s
500:	learn: 19228.4718275	test: 26805.1553083	best: 26805.1553083 (500)	total: 15.5s	remaining: 1m 48s
600:	learn: 18936.5674235	test: 26619.3806212	best: 26619.3806212 (600)	total: 18.8s	remaining: 1m 46s
700:	learn: 18652.1225602	test: 26457.6689847	best: 26457.6689847 (700)	total: 22.1s	remaining: 1m 44s
800:	learn: 18381.0647430	test: 26318.9151599	best: 26318.9151599 (800)	total: 25.6s	remaining: 1m 42s
900:	learn: 18148.8239082	test: 26179.0375522	best: 26179.0375522 (900)	tota

Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 46448.3809312	test: 48743.6558140	best: 48743.6558140 (0)	total: 20.6ms	remaining: 1m 22s
100:	learn: 37919.4109831	test: 42094.8372093	best: 42094.8372093 (100)	total: 2.14s	remaining: 1m 22s
200:	learn: 36146.9362969	test: 41645.6372093	best: 41634.3162791 (184)	total: 4.12s	remaining: 1m 17s
bestTest = 41584.94884
bestIteration = 248
Shrink model to first 249 iterations.
regime 2: n=3449


Default metric period is 5 because MAE is/are not implemented for GPU


0:	learn: 212703.5052823	test: 215517.0552147	best: 215517.0552147 (0)	total: 16.7ms	remaining: 1m 6s
100:	learn: 144794.4176964	test: 176189.5460123	best: 176138.4171779 (99)	total: 1.63s	remaining: 1m 3s
200:	learn: 131343.1435656	test: 171973.5092025	best: 171973.5092025 (200)	total: 3.21s	remaining: 1m
bestTest = 171615.0307
bestIteration = 208
Shrink model to first 209 iterations.
regime 3: n=1290


In [27]:
def predict_mixture(X, classifier, regressors_dict, regime_ids):
    proba = classifier.predict_proba(X)
    preds = np.column_stack([regressors_dict[i].predict(X) for i in regime_ids])
    return (proba * preds).sum(axis=1)

In [28]:
def wmae(y_true, y_pred, weight):
    return np.average(np.abs(y_true - y_pred), weights=weight)

In [29]:
pred_train_val = predict_mixture(X_val, clf_4_tuned, regressors_4, [0,1,2,3])
print(wmae(y_val, pred_train_val, w_val))

59832.463526426196


In [33]:
def weighted_median(values, weights):
    order = np.argsort(values)
    v, wt = np.asarray(values)[order], np.asarray(weights)[order]
    return v[np.searchsorted(np.cumsum(wt), wt.sum() / 2)]

In [34]:
def affine_cal(y_t, p, wt, lo=0.90, hi=1.15, n=51):
    best = (np.inf, 1.0, 0.0)
    for s in np.linspace(lo, hi, n):
        sh = weighted_median(y_t - s * p, wt)
        sc = wmae(y_t, np.clip(s * p + sh, y_t.min(), y_t.max()), wt)
        if sc < best[0]:
            best = (sc, float(s), float(sh))
    return best

In [35]:
cal_wmae, CAL_SCALE, CAL_SHIFT = affine_cal(y_val.to_numpy(), pred_train_val, w_val.to_numpy())

In [36]:
print(f'До калибровки: {wmae(y_val, pred_train_val, w_val):.2f}')
print(f'После калибровки: {cal_wmae:.2f} (scale={CAL_SCALE:.4f}, shift={CAL_SHIFT:.0f})')

До калибровки: 59832.46
После калибровки: 59575.69 (scale=1.0000, shift=-3578)


In [43]:
val_df = pd.DataFrame({
    'target': y_val,
    'w': w_val,
    'pred': pred_train_val
}, index=X_val.index)

val_df['abs_error'] = (val_df['target'] - val_df['pred']).abs()
val_df['weighted_error'] = val_df['abs_error'] * val_df['w']

bins = [0, 50000, 83000, 150000, 300000, 500000, 1000000, np.inf]
val_df['target_segment'] = pd.cut(val_df['target'], bins=bins)

In [44]:
val_df['signed_error'] = val_df['pred'] - val_df['target']

bias_report = val_df.groupby('target_segment', observed=True).agg(
    n=('target', 'size'),
    mean_target=('target', 'mean'),
    mean_pred=('pred', 'mean'),
    mean_signed_error=('signed_error', 'mean'),
    median_signed_error=('signed_error', 'median'),
).assign(
    relative_bias=lambda d: d['mean_signed_error'] / d['mean_target']
)

bias_report_plot = bias_report.reset_index()
bias_report_plot['target_segment'] = bias_report_plot['target_segment'].astype(str)

print(bias_report_plot)

fig = px.bar(
    bias_report_plot, x='target_segment', y='relative_bias',
    title='Относительное смещение предсказаний по сегментам (>0 = завышение, <0 = занижение)',
    text_auto='.1%'
)

fig.update_layout(width=900, height=450, xaxis_title='Сегмент target', yaxis_title='Относительное смещение')

fig.show()

          target_segment     n   mean_target      mean_pred  \
0         (0.0, 50000.0]  5674  3.350531e+04   55284.128905   
1     (50000.0, 83000.0]  5040  6.442104e+04   75079.373271   
2    (83000.0, 150000.0]  3534  1.081247e+05  114825.782716   
3   (150000.0, 300000.0]  1384  2.020433e+05  190473.985900   
4   (300000.0, 500000.0]   379  3.771418e+05  321526.036749   
5  (500000.0, 1000000.0]   153  6.575788e+05  512787.006746   
6       (1000000.0, inf]    50  1.207344e+06  746676.694250   

   mean_signed_error  median_signed_error  relative_bias  
0       21778.816564         10861.332960       0.650011  
1       10658.334632           374.373407       0.165448  
2        6701.050232          1596.733137       0.061975  
3      -11569.318615        -11491.502347      -0.057262  
4      -55615.726590        -61995.690889      -0.147466  
5     -144791.802255       -146013.674165      -0.220189  
6     -460666.900211       -351988.452544      -0.381554  


# Submit

In [42]:
Y = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns])

Y['salary_6to12m_avg_missing'] = Y['salary_6to12m_avg'].isna().astype(int)

mask_missing_test = Y['salary_6to12m_avg'].isna()
print(f'Пропусков salary_6to12m_avg в тесте: {mask_missing_test.sum()} ({mask_missing_test.mean():.1%})')

for col in cat_cols:
    Y[col] = Y[col].fillna('missing').astype(str)

if mask_missing_test.sum() > 0:
    Y_impute_input = Y.loc[mask_missing_test, imputer_features]
    imputed_test_values = salary_imputer_model.predict(Y_impute_input)
    Y.loc[mask_missing_test, 'salary_6to12m_avg'] = imputed_test_values

pred_test = predict_mixture(Y, clf_4_tuned, regressors_4, [0,1,2,3])

month_h_map = {'2024-07-31': 1, '2024-08-31': 2, '2024-09-30': 3, '2024-10-31': 4, '2024-11-30': 5}
RATE = 0.01  # стартовое значение, можно потом подобрать точнее

h = test_df['dt'].astype(str).map(month_h_map).to_numpy()
prog_scale = CAL_SCALE + RATE * (h - 1)

pred_test_calibrated = np.clip(
    prog_scale * pred_test + CAL_SHIFT,
    y_train.min(), y_train.max()   # клип по диапазону, наблюдаемому в train
)

print('До калибровки (test), среднее:', pred_test.mean())
print('После калибровки (test), среднее:', pred_test_calibrated.mean())

Y['predict'] = pred_test_calibrated
Y['id'] = test_df['id']
Y[['id', 'predict']].set_index('id').to_csv('submit_catboost.csv', decimal=',', sep=';')

C:\Users\Admin\AppData\Local\Temp\ipykernel_19388\1438355977.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  Y['salary_6to12m_avg_missing'] = Y['salary_6to12m_avg'].isna().astype(int)


Пропусков salary_6to12m_avg в тесте: 65580 (89.6%)
До калибровки (test), среднее: 104018.82011259662
После калибровки (test), среднее: 102425.14195306705


C:\Users\Admin\AppData\Local\Temp\ipykernel_19388\1438355977.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  Y['predict'] = pred_test_calibrated
C:\Users\Admin\AppData\Local\Temp\ipykernel_19388\1438355977.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  Y['id'] = test_df['id']
